# NB04 — Satellite prediction of Musanga cover fraction from UAV mosaics (V3)

**Objective:** predict Musanga canopy fraction from year-matched Sentinel-2 and annual Google Satellite Embeddings, train across forest sites and evaluate independent site transfer, blocked within-site extrapolation, and annotated-circle transfer at multiple spatial supports.

**Sources:** new frozen NB03 native-RGB M2 full probability maps; V2/08 UAV-acquisition years; V2 native Sentinel-2 10 m projection registry and satellite feature definitions. Source UAV probabilities are *model-derived pseudo-labels*, not independent ground truth. The manually annotated 30 V3 exhaustive circles are an additional diagnostic, but the final M2 that produced pseudo-labels was trained on these circles, so independence is from satellite-model training only, **not** from UAV-model training. Never claim circle comparisons validate UAV probabilities independently.

**Acquisition years from V2:** Baego 2023, Yoko 2020, Yangambi 2024, MOS 2024, optional Luki 2023. **Assumption to verify:** both MOS 3–4 and MOS 5–6 were acquired in the same year as MOS 1–2. Do not interpret within-year temporality as established. Year−1/year/year+1 predictors are retrospective and may contain post-UAV imagery.

**Workflow:** 00 manifest and years → 01 exact S2-aligned grids → 02 begin year-specific satellite downloads (independent of NB03) → 03 aggregate full-UAV probability and thresholded cover with direct source-to-grid area support and coverage masks at 10/20/30/50/75/100 m → 04 build matched predictor/target samples at fixed scales → 05 leave-one-site-out and spatial-block model comparison → 06 independent-annotation-circle diagnostics → 07 diagnostic figures, audit, and best-model refit for satellite maps. Blocks requiring NB03 products stop with an actionable missing-file list; you can download satellite inputs while NB03 runs.

**Analysis distinction:** mean UAV Musanga probability (expected cover) and area fraction of pixels with probability≥0.5 (threshold cover) are separate responses, always with UAV-valid-area fraction. Evaluate both, never label either as field-measured cover. Exclude cells with <90% UAV coverage and near-zero S2 feature support; retain coverage diagnostics. Explicit physical areas and aligned grids are essential, especially 75 m which must be calculated directly (not by stacking 10 m rasters).

**Evaluation contract:** all comparisons use matched geographic cells per scale/family (intersection support); report per-site errors and whole-mosaic maps, and the 30 circle-level observed cover fractions separately. Never randomly split neighboring 10 m pixels: primary leave-one-site-out (Baego/MOS/Yangambi/Yoko; all three MOS mosaics belong to one group); secondary blocked ~500 m grid with ≥100 m spatial buffer and no leakage among overlapping scales. Circle reference polygons and buffers are **excluded from satellite model training** when using pseudo-targets; for site-LOSO, complete held-out site is excluded. Rank models using outer-fold predictions only.

**Start here:** Run setup, grid registry and download cells first. Google Earth Engine authentication must be performed interactively in your Colab account; this notebook cannot authenticate or launch cloud jobs by itself. Downloads are configurable, checkpointed, and never overwrite existing verified raster files.


In [ ]:
%pip -q install earthengine-api geemap geedim rasterio geopandas shapely scikit-learn pandas matplotlib scipy tqdm
import json,os,math,hashlib,shutil,time,warnings
from pathlib import Path
import numpy as np,pandas as pd,rasterio
from rasterio.windows import Window
from rasterio.warp import reproject,Resampling,transform_bounds
from affine import Affine
from IPython.display import display
from google.colab import drive
if not Path("/content/drive/MyDrive").exists():drive.mount("/content/drive")
ROOT=next((p for p in [Path("/content/drive/MyDrive/Colab Notebooks/Musanga/Final"),
                         Path("/content/drive/MyDrive/Musanga/Final")] if p.is_dir()),None)
assert ROOT is not None
LEGACY=ROOT.parent/"KVO_work/KVO_08_satellite_fraction"
LEGACY_GRIDS=LEGACY/"validation/09_s2_canonical_10m_grids.json"
MANIFEST=ROOT/"Outputs/V3/00_input_audit/00_input_manifest.json"
assert MANIFEST.exists(),MANIFEST
A=json.loads(MANIFEST.read_text())
assert A.get("schema_version")=="v3_input_audit_5_nadine_circles"
SOURCES={m["file"]:dict(site=m["site"],path=Path(m["path"])) for m in A["mosaics"]}
assert len(SOURCES)>=6 and all(q["path"].exists() for q in SOURCES.values())
YEARS={"Baego":2023,"Yoko":2020,"Yangambi":2024,"MOS":2024,"Luki":2023}
PROVENANCE={"Baego":"V2/08","Yoko":"V2/08","Yangambi":"V2/08",
            "MOS":"V2/08 for MOS_1_2; extrapolated provisionally to MOS_3_4/MOS_5_6",
            "Luki":"V2/08; exact acquisition date unresolved"}
INCLUDE_LUKI=False
if INCLUDE_LUKI:
    p=ROOT.parent/"data/Geo_Ref/Luki.tif"
    assert p.exists()
    SOURCES["Luki.tif"]=dict(site="Luki",path=p)
SITES=sorted(set(x["site"] for x in SOURCES.values()))
OUT=ROOT/"Outputs/V3/04_satellite_Musanga"
GRID_DIR=OUT/"grid";FEAT_DIR=OUT/"features";AGG_DIR=OUT/"uav_aggregated"
MODEL_DIR=OUT/"models";QA_DIR=OUT/"qa"
for p in (GRID_DIR,FEAT_DIR,AGG_DIR,MODEL_DIR,QA_DIR):p.mkdir(parents=True,exist_ok=True)
MODE="final_all30"   # NB03 model, must match native RGB probability maps
NB03=ROOT/("Outputs/V3/03_M2_full_mosaic_"+MODE)/"probability_maps"
THRESHOLD=0.5
MIN_UAV_COVERAGE=.90
MIN_FEATURE_VALID=.90
SCALES=[10,20,30,50,75,100]
assert set(SCALES)=={10,20,30,50,75,100}
assert all(x["site"] in YEARS for x in SOURCES.values())
reg=[]
for m,x in SOURCES.items():
    with rasterio.open(x["path"]) as d:
        assert d.crs is not None
        reg.append(dict(mosaic=m,site=x["site"],year=YEARS[x["site"]],
                        crs=str(d.crs),width=d.width,height=d.height,
                        raster_GB=round(x["path"].stat().st_size/1e9,3),
                        year_basis=PROVENANCE[x["site"]]))
display(pd.DataFrame(reg))
(OUT/"04_input_registry.json").write_text(json.dumps(
    dict(years=YEARS,provenance=PROVENANCE,sources={k:dict(site=v["site"],path=str(v["path"]))
    for k,v in SOURCES.items()},uav_model=MODE),indent=2))
print("NB03 files may still be running. Satellite download blocks do not require NB03.")


## 01 — Canonical 10 m satellite grid and independent 75 m aggregation grids

Use V2's **actual native S2 B2 tile CRS/origin** wherever available. For MOS multi-mosaic analysis, extend a single MOS S2 grid to the union of the mosaics and retain MOS as one cross-validation site. Coarser grids inherit the same UTM origin; 75 m is independent and never formed by averaging prefiltered 10 m pixels.


In [ ]:
assert LEGACY_GRIDS.exists(),(
    "V2/08 native Sentinel-2 grid registry missing; build it in V2/08 Block 02 first",LEGACY_GRIDS)
old=json.loads(LEGACY_GRIDS.read_text())
SITE_V2={"Baego":"Baego","Yangambi":"Yangambi","Yoko":"Yoko",
         "MOS":"MOS_1_2","Luki":"Luki"}
GRIDS={}
for site in SITES:
    key=SITE_V2[site]
    assert key in old,("Missing V2 grid",site,key)
    g=old[key]
    assert int(g["uav_year"])==YEARS[site],(site,g["uav_year"])
    crs=g["uav_crs"]
    # ORIGINAL native scene grid: universal origin for *all* output scales,
    # independent of each site's mosaic footprint.
    native=Affine(*g["native_scene_transform"])
    assert abs(native.a-10)<1e-6 and abs(native.e+10)<1e-6
    assert abs(native.b)<1e-9 and abs(native.d)<1e-9
    rasters=[q["path"] for q in SOURCES.values() if q["site"]==site]
    bounds=[]
    for p in rasters:
        with rasterio.open(p) as ds:
            assert str(ds.crs)==crs,("Mosaic and canonical S2 CRS differ",site,p,ds.crs,crs)
            bounds.append(ds.bounds)
    west=min(b.left for b in bounds);south=min(b.bottom for b in bounds)
    east=max(b.right for b in bounds);north=max(b.top for b in bounds)
    scale=10
    x0=native.c+math.floor((west-native.c)/scale)*scale
    y0=native.f-math.floor((native.f-north)/scale)*scale
    nx=math.ceil((east-x0)/scale);ny=math.ceil((y0-south)/scale)
    GRIDS[site]=dict(site=site,year=YEARS[site],crs=crs,
        native_scene_transform=list(native)[:6],transform=[10,0,x0,0,-10,y0],
        width=nx,height=ny,bounds=[west,south,east,north],
        mosaics=[k for k,q in SOURCES.items() if q["site"]==site],
        scale=10,s2_native_tile=g["s2_native_tile"])
    print(site,"S2 native CRS",crs,"grid",nx,ny,"year",YEARS[site],
          "tile",g["s2_native_tile"])
(OUT/"04_canonical_site_grids.json").write_text(json.dumps(GRIDS,indent=2))

def scaled_grid(site,scale):
    """Exact nested physical UTM grid; does not assume scale divides 10."""
    g=GRIDS[site];native=Affine(*g["native_scene_transform"])
    west,south,east,north=g["bounds"]
    x0=native.c+math.floor((west-native.c)/scale)*scale
    y0=native.f-math.floor((native.f-north)/scale)*scale
    nx=math.ceil((east-x0)/scale);ny=math.ceil((y0-south)/scale)
    return dict(site=site,scale_m=scale,crs=g["crs"],
                transform=[scale,0,x0,0,-scale,y0],width=nx,height=ny)
for scale in SCALES:
    assert all(scaled_grid(s,scale)["width"]>0 for s in SITES)
print("Canonical grids ready. Features can now be downloaded independently of NB03.")


## 02 — Download annual S2, seasonal S2 and Google annual embeddings (year-matched)

**This block can run while NB03 is producing UAV maps.** Earth Engine access is through the user's authenticated Google Cloud project `musanga-511110`, as in V2. To avoid unexpectedly launching a very large download, default to **Baego annual S2 + annual embeddings** for the first run; add more sites/families after examining QA. The generated outputs are 10 m float32 GeoTIFFs on each site's exact canonical S2 grid, with -9999 NoData, band descriptions and immutable fingerprint/provenance. Existing verified files are reused; incompatible files cause an error rather than overwrite. The download is direct to Colab scratch before verified copy to Drive, using geemap/geedim (V2 approach).

Families: RAW_ANNUAL (10 reflectances + NDVI, NDMI, NBR, NDRE + valid-count), S2_QUARTERLY (4 × 10 reflectances + counts), EMB_ANNUAL (64 Google annual embeddings). Three-year retrospective S2 and embedding families can be added once first acquisition-year exports succeed; do not use future-year imagery in prospective monitoring. No BRDF correction in this first benchmark.


In [ ]:
import ee,geemap,geedim
EE_PROJECT="musanga-511110"
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
S2_BANDS=["B2","B3","B4","B8","B5","B6","B7","B8A","B11","B12"]
S2_INDEX=["NDVI","NDMI","NBR","NDRE"]
EMB_BANDS=[f"A{k:02d}" for k in range(64)]
FAMILIES={
 "RAW_ANNUAL":S2_BANDS+S2_INDEX+["N_CLEAR_S2"],
 "S2_QUARTERLY":[f"Q{q}_{band}" for q in range(1,5)
                 for band in S2_BANDS+["N_CLEAR"]],
 "EMB_ANNUAL":EMB_BANDS
}
S2_SOURCE="COPERNICUS/S2_SR_HARMONIZED"
EMB_SOURCE="GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL"
SCL_REJECT=[0,1,3,7,8,9,10,11]
def region(site):
    g=GRIDS[site]
    from rasterio.warp import transform_bounds
    aff=Affine(*g["transform"])
    west=aff.c;south=aff.f-g["height"]*10
    east=aff.c+g["width"]*10;north=aff.f
    b=transform_bounds(g["crs"],"EPSG:4326",west,south,east,north,densify_pts=21)
    return ee.Geometry.Rectangle(list(b),proj="EPSG:4326",geodesic=False)
def clean_s2(im):
    scl=im.select("SCL")
    valid=ee.Image(1)
    for x in SCL_REJECT:valid=valid.And(scl.neq(x))
    valid=valid.And(im.select("B8A").mask()).And(im.select("B9").mask())
    # Preserve native reflectances until post-composite 0.0001 scaling.
    out=ee.Image.cat([im.select(S2_BANDS[:4]),
                      im.select(S2_BANDS[4:]).resample("bilinear")])
    return out.updateMask(valid).copyProperties(im,["system:time_start"])
def add_indices(im):
    return ee.Image.cat([im,
        im.normalizedDifference(["B8","B4"]).rename("NDVI"),
        im.normalizedDifference(["B8","B11"]).rename("NDMI"),
        im.normalizedDifference(["B8","B12"]).rename("NBR"),
        im.normalizedDifference(["B8A","B5"]).rename("NDRE")])
def make_satellite_image(site,family):
    yr=YEARS[site]
    area=region(site)
    if family=="EMB_ANNUAL":
        col=(ee.ImageCollection(EMB_SOURCE)
            .filterDate(f"{yr}-01-01",f"{yr+1}-01-01").filterBounds(area))
        n=int(col.size().getInfo())
        assert n>0,(site,yr,"Annual embedding unavailable")
        return col.mosaic().select(EMB_BANDS).toFloat().clip(area)
    s2=(ee.ImageCollection(S2_SOURCE)
        .filterDate(f"{yr}-01-01",f"{yr+1}-01-01")
        .filterBounds(area).map(clean_s2))
    assert int(s2.size().getInfo())>0,(site,yr,"No S2 SR scenes")
    if family=="RAW_ANNUAL":
        med=s2.median().multiply(.0001)
        count=s2.select("B2").count().unmask(0).toFloat().rename("N_CLEAR_S2")
        return ee.Image.cat([add_indices(med),count]).toFloat().clip(area)
    if family=="S2_QUARTERLY":
        layers=[]
        for quarter in (1,2,3,4):
            first=f"{yr}-{quarter*3-2:02d}-01"
            last=f"{yr+1}-01-01" if quarter==4 else f"{yr}-{quarter*3+1:02d}-01"
            col=s2.filterDate(first,last)
            assert int(col.size().getInfo())>0,(site,yr,quarter,"No seasonal scenes")
            med=col.median().multiply(.0001)
            layers.append(med.rename([f"Q{quarter}_{b}" for b in S2_BANDS]))
            layers.append(col.select("B2").count().unmask(0).toFloat().rename(f"Q{quarter}_N_CLEAR"))
        return ee.Image.cat(layers).toFloat().clip(area)
    raise ValueError(family)

# FIRST RUN: one small controlled feature batch. Expand after success.
DOWNLOAD_SITES=["Baego"]             # expand to SITES for full study
DOWNLOAD_FAMILIES=["RAW_ANNUAL","EMB_ANNUAL"]  # then S2_QUARTERLY
START_DOWNLOAD=True                    # execute to download; requires Colab+EE auth
assert set(DOWNLOAD_SITES).issubset(SITES)
assert set(DOWNLOAD_FAMILIES).issubset(FAMILIES)
LOCAL_DOWNLOAD=Path("/content/nb04_satellite_download")
LOCAL_DOWNLOAD.mkdir(exist_ok=True)
def feature_path(site,family):
    return FEAT_DIR/family/f"{site}_{family}_{YEARS[site]}_10m.tif"
def feature_ok(path,site,family):
    if not path.exists():return False
    g=GRIDS[site]
    try:
        with rasterio.open(path) as ds:
            return (ds.count==len(FAMILIES[family]) and ds.width==g["width"] and
                ds.height==g["height"] and str(ds.crs)==g["crs"] and
                ds.transform.almost_equals(Affine(*g["transform"]),precision=7) and
                ds.nodata==-9999 and ds.dtypes[0]=="float32" and
                ds.tags().get("FEATURE_FAMILY")==family and
                ds.tags().get("ACQUISITION_YEAR")==str(YEARS[site]))
    except Exception:return False
def download_feature(site,family):
    target=feature_path(site,family)
    target.parent.mkdir(exist_ok=True)
    if target.exists():
        assert feature_ok(target,site,family),("Incompatible old feature; inspect",target)
        print("CACHE HIT",site,family,target,flush=True)
        return
    g=GRIDS[site]
    tmp=LOCAL_DOWNLOAD/f"{site}_{family}_{YEARS[site]}.partial.tif"
    pending=target.with_name(target.name+".uploading")
    assert not pending.exists(),("Interrupted Drive upload; inspect",pending)
    if not tmp.exists():
        image=make_satellite_image(site,family).unmask(-9999,sameFootprint=False).toFloat()
        print("DOWNLOADING",site,family,len(FAMILIES[family]),"bands; grid",
              g["width"],"x",g["height"],flush=True)
        geemap.download_ee_image(
            image,filename=str(tmp),crs=g["crs"],crs_transform=g["transform"],
            shape=(g["height"],g["width"]),dtype="float32",
            resampling="near",max_tile_dim=256,num_threads=1,overwrite=False)
    # Validate *full raster content* and georeferencing before publication.
    with rasterio.open(tmp,"r+") as ds:
        assert ds.count==len(FAMILIES[family]),(site,family,ds.count)
        assert (ds.width,ds.height)==(g["width"],g["height"])
        assert str(ds.crs)==g["crs"] and ds.transform.almost_equals(
            Affine(*g["transform"]),precision=7)
        assert ds.dtypes[0]=="float32"
        for band in range(1,ds.count+1):
            for _,window in ds.block_windows(band):
                val=ds.read(band,window=window)
                assert np.isfinite(val).all(),("NaN/Inf in EE image",site,family,band)
            ds.set_band_description(band,FAMILIES[family][band-1])
        assert ds.nodata in (None,-9999) or (
            ds.nodata is not None and np.isneginf(ds.nodata)),(
            "Unexpected EE NoData",ds.nodata)
        ds.nodata=-9999.
        ds.update_tags(FEATURE_FAMILY=family,ACQUISITION_YEAR=str(YEARS[site]),
                       SOURCE_COLLECTION=S2_SOURCE if family!="EMB_ANNUAL" else EMB_SOURCE,
                       UTM_GRID_SOURCE="V2_08_native_S2_B2",BRDF="NONE",
                       FEATURE_BANDS=json.dumps(FAMILIES[family]))
    assert feature_ok(tmp,site,family)
    shutil.copyfile(tmp,pending)
    assert feature_ok(pending,site,family)
    os.replace(pending,target)
    tmp.unlink(missing_ok=True)
    print("SAVED",site,family,target,flush=True)
if START_DOWNLOAD:
    tasks=[(s,f) for s in DOWNLOAD_SITES for f in DOWNLOAD_FAMILIES]
    for j,(site,family) in enumerate(tasks,1):
        print(f"[{j}/{len(tasks)}] {site} {family}",flush=True)
        download_feature(site,family)
print("Available verified satellite rasters:")
display(pd.DataFrame([dict(site=s,family=f,available=feature_ok(feature_path(s,f),s,f))
                      for s in SITES for f in FAMILIES]))


## 03 — Direct native-UAV → target-scale aggregation (10, 20, 30, 50, 75, 100 m)

**Only run when NB03's full-mosaic probability products are complete.** Aggregation is area-weighted at the target physical resolution by GDAL's `average` resampling. It does **not** average already aggregated/gated 10 m references. The output for every mosaic×scale contains four float32 bands: mean Musanga probability, fraction of valid native UAV pixels with probability ≥0.50, fraction of total physical cell covered by valid UAV RGB, and within-cell probability standard deviation. Cells with coverage below 90% remain in the product with the coverage diagnostic but are excluded from model sampling. The 75 m raster is created directly on its own S2-origin-aligned grid.

Original UAV probabilities are only predictions from the final native M2, never species polygons or ground-truth fractional cover. MOS 3–4 is an inference-only region; mark it as such and assess transfer carefully. Reprojection uses on-disk derived masks to avoid loading enormous UAV images into Colab RAM. Nothing modifies the source imagery, model outputs, or cached satellite features.


In [ ]:
from rasterio.features import rasterize
from rasterio.warp import reproject
NODATA_U16=65535
SCALE_U16=10000
TARGET_NAMES=["p_mean","cover_fraction_p05","uav_valid_fraction","p_sd"]
WORK=Path("/content/nb04_aggregate_work");WORK.mkdir(exist_ok=True)
RUN_AGGREGATION=True
AGG_SITES=list(SITES)  # optionally ['Baego'] for a first pilot

def uav_probability_file(name):
    stem=Path(name).stem
    return NB03/f"{stem}_M2_{MODE}_probability_u16_FULL.tif"

def agg_file(mosaic,scale):
    return AGG_DIR/f"{Path(mosaic).stem}_M2_{MODE}_{scale}m.tif"

def target_spec(name,scale,src):
    grid=scaled_grid(SOURCES[name]["site"],scale)
    return dict(mosaic=name,site=SOURCES[name]["site"],scale_m=scale,
        source_path=str(uav_probability_file(name)),
        source_key=src.tags().get("SPEC"),target_grid=grid,
        threshold=THRESHOLD,uav_scale=SCALE_U16,uav_nodata=NODATA_U16,
        pipeline="NB04_native_direct_area_average_v1")

def band_profile(src,dtype,nodata):
    kw=src.profile.copy()
    for field in ("photometric","alpha","extrasamples","NBITS","colorinterp","interleave"):
        kw.pop(field,None)
    kw.update(driver="GTiff",count=1,dtype=dtype,nodata=nodata,
              tiled=True,blockxsize=256,blockysize=256,compress="deflate",
              bigtiff="YES",SPARSE_OK=False)
    return kw

def create_derived_native_layers(src,workdir):
    """Temporary on-disk 0/1 validity and native p≥0.5 masks.

    Invalid RGB is 0 in validity; invalid is 255 (NoData) in cover.
    Sources are processed in native block windows, never held entirely in RAM.
    """
    valid_path=workdir/"valid_native.tif"
    cover_path=workdir/"cover_native.tif"
    sq_path=workdir/"probability_squared_native.tif"
    valid_profile=band_profile(src,"uint8",None)
    cover_profile=band_profile(src,"uint8",255)
    sq_profile=band_profile(src,"float32",-9999)
    with rasterio.open(valid_path,"w",**valid_profile) as vd,\
         rasterio.open(cover_path,"w",**cover_profile) as cv,\
         rasterio.open(sq_path,"w",**sq_profile) as sq:
        for _,window in src.block_windows(1):
            a=src.read(1,window=window)
            ok=a!=NODATA_U16
            vd.write(ok.astype("uint8"),1,window=window)
            mask=np.full(a.shape,255,"uint8")
            mask[ok]=(a[ok]>=round(THRESHOLD*SCALE_U16)).astype("uint8")
            cv.write(mask,1,window=window)
            p2=np.full(a.shape,-9999,"float32")
            p2[ok]=(a[ok].astype("float32")/SCALE_U16)**2
            sq.write(p2,1,window=window)
    return valid_path,cover_path,sq_path

def aggregate_one(name,scale):
    source=uav_probability_file(name)
    assert source.exists(),("NB03 not finished for this source",name,source)
    output=agg_file(name,scale)
    with rasterio.open(source) as src:
        assert src.dtypes[0]=="uint16" and src.nodata==NODATA_U16,(name,src.dtypes,src.nodata)
        assert src.tags().get("COMPLETE")=="YES",(name,"NB03 output not finalized")
        spec=target_spec(name,scale,src)
        spec_sha=hashlib.sha256(json.dumps(spec,sort_keys=True).encode()).hexdigest()
        grid=spec["target_grid"]
        trans=Affine(*grid["transform"])
        if output.exists():
            with rasterio.open(output) as old:
                assert old.tags().get("SPEC_SHA")==spec_sha and old.count==4
            print("AGG CACHE HIT",name,scale,flush=True)
            return output
        sitework=WORK/Path(name).stem
        sitework.mkdir(exist_ok=True)
        # Reuse temporary native masks across scales in the current session.
        validfile,coverfile,sqfile=[sitework/x for x in (
            "valid_native.tif","cover_native.tif","probability_squared_native.tif")]
        if not all(p.exists() for p in (validfile,coverfile,sqfile)):
            for p in (validfile,coverfile,sqfile):p.unlink(missing_ok=True)
            create_derived_native_layers(src,sitework)
        assert all(p.exists() for p in (validfile,coverfile,sqfile))
        shape=(grid["height"],grid["width"])
        assert shape[0]*shape[1]<150_000_000,("Oversized grid: do spatial chunking",name,scale,shape)
        def warp(source_band,source_nodata,dest_nodata=-9999):
            dest=np.full(shape,dest_nodata,"float32")
            reproject(source=source_band,destination=dest,
                src_transform=src.transform,src_crs=src.crs,
                src_nodata=source_nodata,dst_transform=trans,dst_crs=grid["crs"],
                dst_nodata=dest_nodata,resampling=Resampling.average,
                num_threads=2,warp_mem_limit=256)
            return dest
        p=warp(rasterio.band(src,1),NODATA_U16)
        p=np.where(p==-9999,-9999,p/SCALE_U16)
        with rasterio.open(validfile) as valid_src,\
             rasterio.open(coverfile) as cover_src,\
             rasterio.open(sqfile) as sq_src:
            coverage=warp(rasterio.band(valid_src,1),None)
            cover=warp(rasterio.band(cover_src,1),255)
            sq=warp(rasterio.band(sq_src,1),-9999)
        coverage=np.clip(coverage,0,1)
        defined=(p!=-9999)&(cover!=-9999)&(sq!=-9999)&(coverage>0)
        sd=np.full(shape,-9999,"float32")
        sd[defined]=np.sqrt(np.maximum(0,sq[defined]-p[defined]**2))
        desc=dict(driver="GTiff",height=shape[0],width=shape[1],count=4,
            crs=grid["crs"],transform=trans,dtype="float32",nodata=-9999,
            tiled=True,blockxsize=256,blockysize=256,compress="deflate",
            predictor=3,bigtiff="YES")
        partial=output.with_name(output.name+".partial.tif")
        with rasterio.open(partial,"w",**desc) as dst:
            for idx,layer in enumerate((p,cover,coverage,sd),1):
                dst.write(layer.astype("float32"),idx)
                dst.set_band_description(idx,TARGET_NAMES[idx-1])
            dst.update_tags(SPEC_SHA=spec_sha,DATA_SOURCE=str(source),
                UAV_PROBABILITY_MODEL=MODE,AGGREGATION="NATIVE_DIRECT_AVERAGE",
                MIN_VALID_COVERAGE=str(MIN_UAV_COVERAGE),COMPLETE="YES")
        assert partial.exists()
        os.replace(partial,output)
        print("AGG SAVED",name,scale,"grid",shape,"supported",
              int(np.sum(defined&(coverage>=MIN_UAV_COVERAGE))),flush=True)
        return output

if RUN_AGGREGATION:
    missing=[str(uav_probability_file(name)) for name in SOURCES
             if SOURCES[name]["site"] in AGG_SITES and not uav_probability_file(name).exists()]
    assert not missing,("NB03 outputs missing: keep downloading satellite features first",missing)
    for name in SOURCES:
        if SOURCES[name]["site"] not in AGG_SITES:continue
        for scale in SCALES:
            aggregate_one(name,scale)
